In [1]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.distributions import Normal 

In [4]:
pip install "gymnasium[box2d]"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\NITROPC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import gymnasium as gym
import numpy as np
import torch
import torch.optim as optim
from torch.distributions import Normal

env = gym.make("LunarLander-v3", continuous=True)

In [6]:
env = gym.make("LunarLander-v3", continuous=True, render_mode="human")

In [7]:
state_dim = 8
action_dim = 2

In [2]:
class ResidualBlock(nn.Module):
    def __init__(self,in_,out_):
        super().__init__()

        self.linear1 = nn.Sequential(
            nn.Linear(in_,out_),
            nn.LayerNorm(out_),
            nn.ReLU(),
        )
        self.linear2 = nn.Sequential(
            nn.Linear(out_,out_),
            nn.LayerNorm(out_),
        )
        self.shortcut = nn.Sequential()
        self.relu = nn.ReLU()

        if in_ != out_:
            self.shortcut = nn.Sequential(
            nn.Linear(in_,out_),
            nn.LayerNorm(out_)
            )
    def forward(self,x):
        identity = self.shortcut(x)
        x = self.linear1(x)
        x = self.linear2(x)
        x = x + identity

        return self.relu(x)
    
            
class Wesker(nn.Module): # A2C <3
    def __init__(self,in_,out_):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_,in_*2),
            ResidualBlock(in_*2,in_*4),
            ResidualBlock(in_*4,in_*8),
        )
        self.decoder = nn.Sequential(
            ResidualBlock(in_*8,in_*4),
            ResidualBlock(in_*4,in_*2),
            ResidualBlock(in_*2,in_),
        )
        self.mean = nn.Linear(in_,out_)
        self.log_std = nn.Parameter(torch.zeros(out_))

        self.critic = nn.Linear(in_,1)

    def forward(self,x):
        x = self.encoder(x)
        x = self.decoder(x)
        mean = self.mean(x)
        std = torch.exp(self.log_std)
        value = self.critic(x)
        return mean,std,value 

In [3]:
gamma = 0.97
lam = 0.93
eps = 0.15
learning_rate = 5e-5
scheduler_t_max = 1000
ppo_epochs = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class Agent:
    def __init__(self,in_,out_,scheduler_episode=scheduler_t_max,learning_rate=learning_rate,gamma=gamma,lam=lam,eps=eps,ppo_epochs=ppo_epochs):
        self.model = Wesker(in_,out_).to(device)
        self.eps = eps 
        self.gamma = gamma 
        self.lam = lam 
        self.ppo_epochs = ppo_epochs
        self.optimizer = optim.AdamW(self.model.parameters(),lr=learning_rate,betas=(0.9,0.95),weight_decay=1e-4) # betas adica 0.9 retine 90% din media aritmetica a weighturilor trecute si 0.95 ii 95% din media patratica type shit a weighturilor trecute 
        
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer,T_max=scheduler_episode,eta_min=1e-5)
# ideea ii urmatoare: A2C nu are torch.no_grad() la ACT pentru ca foloseste ce primeste din act in update direct,dar PPO le foloeste ca si calcule cu new log probs calculate atunci pe loc ,practic ca si un new - old si da 
    def act(self,state):
        with torch.no_grad():
            state = torch.tensor([state],dtype=torch.float32).to(device)
            mean,std,value = self.model(state)
            dist = Normal(mean,std)
            action = dist.sample() 

            log_probs = dist.log_prob(action).sum(-1)
            #entropy = dist.entropy().sum(-1)
        return (
            action.squeeze(0).cpu().numpy(),
            log_probs,
            value.squeeze()
        )
    
    def gae(self,rewards,values,last_next_value,dones): # k idee toate astea s doar formule implementate :O👅
        values = values + [last_next_value]
        advantage = []
        gae = 0
        # si mergem de la final spre inceput ca gen ii ca si cum mergi de la infinit spre 0 pentru ca daca esti 0 ai reawrdul pana la infinit,daca esti la infinit practic ai doar 0,gen idk daca are logica 
        # but ig it makes sense ?idk 
        for i in reversed(range(len(rewards))):
            delta = rewards[i] + self.gamma * (1-dones[i]) * values[i+1] - values[i]
            gae = delta + gae * self.gamma * self.lam * (1-dones[i])
            advantage.insert(0,gae)

        returned = [v + a for v,a in zip(values[:-1],advantage)]

        return torch.stack(returned).to(device),torch.stack(advantage).to(device)
    def step(self,states,old_log_probs,returned,advantage,actions):
        states = torch.tensor(states,dtype=torch.float32).to(device)
        actions = torch.tensor(actions,dtype=torch.float32).to(device)
        old_log_probs = torch.stack(old_log_probs).detach().view(-1).to(device)
        returned = returned.detach()
        advantage = advantage.detach()
        # norm advantage :
        self.model.train()
        advantage = (advantage - advantage.mean())/(advantage.std() + 1e-8)
        batch_size = len(states)
        mini_batch_size = max(1,batch_size//4)

        for i in range(self.ppo_epochs): # ppo epochs 
            permutari= torch.randperm(batch_size)
            for j in range(0,batch_size,mini_batch_size):
                idx = permutari[j:j+mini_batch_size]

                m_old_log = old_log_probs[idx]
                m_returned = returned[idx]
                m_advantage = advantage[idx]
                m_actions = actions[idx]
                m_states = states[idx]

                # ... 
                
                mean,std,value = self.model(m_states)
                
                dist = Normal(mean,std)
                log_probs = dist.log_prob(m_actions).sum(-1)
                entropy = dist.entropy().sum(-1)
           
                ratio = torch.exp(log_probs - m_old_log)

                surr1 = ratio * m_advantage
                surr2 = torch.clamp(ratio,1-self.eps,1+self.eps)*m_advantage

                actor_loss = -torch.min(surr1,surr2).mean()
                critic_loss = (m_returned - value.squeeze()).pow(2).mean()

                self.optimizer.zero_grad()
                loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy.mean()

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(),0.5)
                self.optimizer.step()
        self.scheduler.step()




        

In [ ]:
agent =Agent(8,2) # this is just an exemple :)))
for episode in range(2000):
    state,info = env.reset()
    total_rewards = 0 
    done = False
    actions = []
    log_probs = []
    values = []
    dones = []
    rewards = []
    states = []

    while not done:
        action,log_prob,value = agent.act(state)
        nextState,reward,terminated,truncated,info = env.step(action)
        done = terminated or truncated
        actions.append(action)
        log_probs.append(log_prob)
        states.append(state)

        values.append(value)
        rewards.append(reward)
        dones.append(done)
        
        if len(rewards) == 64 or done:# ig
            with torch.no_grad():
                if done:
                    last_next_value = torch.tensor(0,dtype=torch.float32).to(device)

                else:
                    ns = torch.tensor([nextState],dtype=torch.float32).to(device)
                    _,_,value = agent.model(ns)
                    last_next_value = value.squeeze()

            returned,advantage = agent.gae(rewards,values,last_next_value,dones)
            agent.step(states,log_probs,returned,advantage,actions)
            actions = []
            log_probs = []
            values = []
            dones = []
            rewards = []
            states = []
        total_rewards += reward
        state = nextState
    print(f"Episode {episode} | reward = {total_rewards:.2f}")



Episode 0 | reward = -104.26
Episode 1 | reward = -165.80
Episode 2 | reward = -497.90
Episode 3 | reward = -654.83
Episode 4 | reward = -543.16
Episode 5 | reward = -614.95
Episode 6 | reward = -547.05
Episode 7 | reward = -511.77
Episode 8 | reward = -389.87
Episode 9 | reward = -828.50
Episode 10 | reward = -750.01
Episode 11 | reward = -249.85
Episode 12 | reward = -449.74
Episode 13 | reward = -560.31
Episode 14 | reward = -591.85
Episode 15 | reward = -621.04
Episode 16 | reward = -644.56
Episode 17 | reward = -393.48
Episode 18 | reward = -980.47
Episode 19 | reward = -301.03
Episode 20 | reward = -592.12


KeyboardInterrupt: 

In [12]:
import importlib
import school_help_env
importlib.reload(school_help_env)

from school_help_env import CrisisToActionCityEnv, run_city_simulator

In [13]:
env = CrisisToActionCityEnv()

agent1 = Agent(
    env.agent_state_dim,
    env.agent_action_dim,
    scheduler_episode=1200,
    learning_rate=5e-5,
    gamma=0.98,
    lam=0.94,
    eps=0.14,
    ppo_epochs=4,
)

agent2 = Agent(
    env.agent_state_dim,
    env.agent_action_dim,
    scheduler_episode=900,
    learning_rate=7e-5,
    gamma=0.96,
    lam=0.92,
    eps=0.18,
    ppo_epochs=4,
)

agent3 = Agent(
    env.agent_state_dim,
    env.agent_action_dim,
    scheduler_episode=1500,
    learning_rate=4e-5,
    gamma=0.99,
    lam=0.95,
    eps=0.12,
    ppo_epochs=5,
)

agent4 = Agent(
    env.agent_state_dim,
    env.agent_action_dim,
    scheduler_episode=800,
    learning_rate=8e-5,
    gamma=0.95,
    lam=0.90,
    eps=0.20,
    ppo_epochs=4,
)

In [ ]:
history = run_city_simulator(
    agent1,
    agent2,
    agent3,
    agent4,
    episodes=5000,
    rollout_len=64,
    seed=42,
    report_every=25,
    narrate=True,
    narrate_every=25,
    narrate_steps=1,
)

Episode 0 | step 0 | shared_sources=0.38 | staff_pool=0.63 | shock=0.00 | season=summer | school_week=0 | weather=0.34 | carbon=251t | reduction=6%
Agent 1 / North Hub | reward=+3.24 | helped=0.51 | unresolved=0.31 | status=active
  actions: resource match=0.91, smart thermostats=0.80, human review=0.63
  effect: matched people to relevant support options; optimized heating and cooling schedules.
  metrics: trust [#######---] 0.66 clarity [#####-----] 0.54 sources [######----] 0.59 | fatigue=0.24 | budget_left=0.70
  weather: disruption=0.45 | transport=0.50 | heat_need=0.08 | cool_need=0.57
  ecology: carbon=231tCO2/yr | reduction=6% | eco_budget_left=$112,973 | forms_saved=10 | trips_saved=4
  safeguards: wrong_guidance=0.08 | missed_urgent=0.03 | human_error=0.33 | recovery=0.22
  user case: Lena, a caregiver -> school_threat_rumor
  assistant: This is a safety rumor that needs source checking before anyone acts on it.
  next steps: Check official school or district channels | Avoid